# Exercise 14: Confirmation Rate

**Source:** [LeetCode 1934](https://leetcode.com/problems/confirmation-rate/description/?envType=study-plan-v2&envId=top-sql-50).

Table: `Signups`
```
+----------------+----------+
| Column Name    | Type     |
+----------------+----------+
| user_id        | int      |
| time_stamp     | datetime |
+----------------+----------+
```
- `user_id` is the column of unique values and
- `time_stamp` is the time of registration.
 

Table: `Confirmations`
```
+----------------+----------+
| Column Name    | Type     |
+----------------+----------+
| user_id        | int      |
| time_stamp     | datetime |
| action         | ENUM     |
+----------------+----------+
```
- `(user_id, time_stamp)` is the primary key.
- `user_id` is a foreign key to `Signups`.
- action is an `ENUM` of types `confirmed` or `timeout`.

Each row is an instance of a `user_id` whose confirmation message was confirmed or expired.

**Task.**
Return the **confirmation rate** (= `confirmed`/(`confirmed` + `timeout`) or `0` if the user did not request any confirmations) and **round it to 2 decimals**. Return the result in **any order**.

Connect to the database `postgres` using the username `postgres`.

In [1]:
%load_ext sql
%sql postgresql+psycopg://postgres@localhost:5432/postgres

The 'toml' package isn't installed. To load settings from pyproject.toml or ~/.jupysql/config, install with: pip install toml

Connecting to 'postgresql+psycopg://postgres@localhost:5432/postgres'

### Create new database for the exercise

Make sure first that a database with the same name is removed in case it already exists.

In [2]:
%sql DROP DATABASE IF EXISTS exercise_14 WITH (FORCE);
%sql CREATE DATABASE exercise_14;

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

Connect to the database.

*Caution.* No semicolon at the end otherwise it belongs to the name of the database.

In [3]:
%sql postgresql+psycopg://postgres@localhost:5432/exercise_14 --alias exercise_14

Connecting and switching to connection 'exercise_14'

#### Fill the database with content

In [16]:
%%sql
-- Drop first tables if existent so that ENUM can be created.
DROP TABLE IF EXISTS Confirmations;
DROP TABLE IF EXISTS Signups;

-- Create the ENUM
DROP TYPE IF EXISTS CONEX;
CREATE TYPE CONEX AS ENUM ('confirmed', 'timeout');

-- Create Tables
CREATE TABLE Signups (
    user_id INTEGER PRIMARY KEY,
    time_stamp TIMESTAMP
);

CREATE TABLE Confirmations (
    user_id INTEGER NOT NULL,
    time_stamp TIMESTAMP NOT NULL,
    action CONEX NOT NULL,
    PRIMARY KEY (user_id, time_stamp),
    FOREIGN KEY (user_id) REFERENCES Signups(user_id)
);

-- Fill tables with content

INSERT INTO Signups (user_id, time_stamp)
VALUES
(3, '2020-03-21 10:16:13'),
(7, '2020-01-04 13:57:59'),
(2, '2020-07-29 23:09:44'),
(6, '2020-12-09 10:39:37');

INSERT INTO Confirmations (user_id, time_stamp, action)
VALUES
(3, '2021-01-06 03:30:46', 'timeout'),
(3, '2021-07-14 14:00:00', 'timeout'),
(7, '2021-06-12 11:57:29', 'confirmed'),
(7, '2021-06-13 12:58:28', 'confirmed'),
(7, '2021-06-14 13:59:27', 'confirmed'),
(2, '2021-01-22 00:00:00', 'confirmed'),
(2, '2021-02-28 23:59:59', 'timeout');

SELECT * FROM Confirmations;

Running query in 'exercise_14'

4 rows affected.

7 rows affected.

7 rows affected.

user_id,time_stamp,action
3,2021-01-06 03:30:46,timeout
3,2021-07-14 14:00:00,timeout
7,2021-06-12 11:57:29,confirmed
7,2021-06-13 12:58:28,confirmed
7,2021-06-14 13:59:27,confirmed
2,2021-01-22 00:00:00,confirmed
2,2021-02-28 23:59:59,timeout


#### Solve the exercise

**Task.**
Return the **confirmation rate** (= `confirmed`/(`confirmed` + `timeout`) or `0` if the user did not request any confirmations) and **round it to 2 decimals**. Return the result in **any order**.

In [17]:
%%sql
SELECT
    S.user_id, COALESCE(R.confirmation_rate, 0) AS confirmation_rate
FROM
    Signups S LEFT JOIN (
        SELECT
            user_id,
            ROUND(COUNT(CASE WHEN action = 'confirmed' THEN 1 END)::NUMERIC/COUNT(*), 2) AS confirmation_rate
        FROM
            Confirmations
        GROUP BY
            user_id
    ) R
ON
    S.user_id = R.user_id;

Running query in 'exercise_14'

4 rows affected.

user_id,confirmation_rate
3,0.00
7,1.00
2,0.50
6,0


#### Explanation

- In a subquery consider `Confirmations` grouped by all users `user_id` who requested (at least) one confirmation.
- To get the number of all confirmed messages do a case count `ROUND(CASE WHEN action = 'confirmed' THEN 1 END)` and divide by the total `COUNT(*)`. Convert into a float using `::NUMERIC` to obtain a float (not an integer) from the division.
- Don't forget to round.
- Join the `Signups` table with a `LEFT JOIN` with the table from the subquery on the `user_id` to add users who never requested a message.
- Taking `COALESCE(confirmation_rate, 0)` replaces the confirmation_rate with `0` if it was `NULL`.

#### Close connection

In [18]:
%sql --close exercise_14
%sql postgresql+psycopg://postgres@localhost:5432/postgres
%sql DROP DATABASE IF EXISTS exercise_14;

Switching to connection 'postgresql+psycopg://postgres@localhost:5432/postgres'

Running query in 'postgresql+psycopg://postgres@localhost:5432/postgres'

++
||
++
++

**Well done!** (approved by Leet)